In [ ]:
%pip -q install gradio pyngrok huggingface_hub fastapi uvicorn pydantic

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Skip the install if it already succeeded in this session
try:
    import llama_cpp
    print("llama-cpp-python already installed:", llama_cpp.__version__)
except ImportError:
    %pip install --only-binary=:all: --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import os, shutil, sqlite3
from huggingface_hub import list_repo_files, hf_hub_download

REPO_ID = "HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive"

# /kaggle/working persists between sessions if you set:
# Session options -> Persistence -> "Files only"  (limit ~20 GB)
BASE_DIR = "/kaggle/working/LocalLLM"
MODEL_DIR = os.path.join(BASE_DIR, "models", "qwen3.6-35b-a3b")
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")
os.makedirs(MODEL_DIR, exist_ok=True)

# ---- Model: reuse if already downloaded, else fetch the XS quant ----
existing = [f for f in os.listdir(MODEL_DIR) if f.endswith(".gguf")]
if existing:
    MODEL_PATH = os.path.join(MODEL_DIR, existing[0])
    print("Model already on disk, skipping download:", MODEL_PATH)
else:
    ggufs = [f for f in list_repo_files(REPO_ID) if f.endswith(".gguf")]
    xs = [f for f in ggufs if "XS" in f.upper()]
    if not xs:
        raise FileNotFoundError(f"No XS quant found. Available: {ggufs}")
    fname = next((f for f in xs if "IQ4_XS" in f.upper()), xs[0])
    print("Downloading", fname, "(~18 GB, first run only)...")
    MODEL_PATH = hf_hub_download(
        repo_id=REPO_ID, filename=fname, local_dir=MODEL_DIR
    )
    shutil.rmtree(os.path.join(MODEL_DIR, ".cache"), ignore_errors=True)  # save space

# ---- Chat DB: create if missing, keep if present ----
with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)
print("Disk used in /kaggle/working: %.1f GB" % (shutil.disk_usage("/kaggle/working").used / 1024**3))

In [ ]:
import sqlite3
import uuid
import os, time, re, subprocess, threading
import gradio as gr
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

# ---------------- 1. Load Interview Instructions & CV Matter ----------------
def load_text_resource(filename, fallback_text):
    search_paths = [
        filename,
        os.path.join(".", filename),
        os.path.join("/kaggle/working", filename),
        os.path.join(BASE_DIR, filename) if "BASE_DIR" in globals() else "",
    ]
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                search_paths.insert(0, os.path.join(root, filename))
    for p in search_paths:
        if p and os.path.isfile(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = f.read().strip()
                if data:
                    print(f"Loaded {filename} from {p} ({len(data)} chars)")
                    return data
            except Exception as e:
                print(f"Error reading {p}: {e}")
    print(f"Using embedded verified memory for {filename}")
    return fallback_text

CV_MATTER = load_text_resource("cvmatter", """Professional Overview
Sasank Navuri
Three-time hackathon winner (NASA Space Apps Hackathon, FOSSEE IIT Bomaby, and MSI Hackathon).
Two internship experience (IIT Bombay, and Exo gear). Active open-source contributor to (OpenCV, opencv-contrib,
and OSDAG), with research experience in quantum technologies.
Education
Vellore Institute of Technology Andhra Pradesh (VIT-AP)
Integrated M.Tech in Software Engineering
Experience
Amaravati, AP
2022– 2027
Intern — IIT Bombay (Hybrid)
Indian Institute of Technology Bombay
May 2025– Jan 2026
Mumbai, MH
Under Prof. Siddhartha Ghosh, contributed to OSDAG by developing real-time 3D simulations using Blender and inte
grating GLB models via Three.js into a React + Django platform.
Multiple Roles Intern — Exogear Startup
Jul 2024– Oct 2024
Bengaluru, KA
Assisted in website deployment, CAD-based prototyping, testing workflows, documentation, marketing, and patent/funding
processes.
Research Experience
Research under Dr. Kankanala Srinivas
Conducted research on quantum-resistant image encryption using chaotic maps for secure image transmission using MAT
LAB and Python.
Projects
OSDAG-Web — React, Django, Docker
Developed interactive 3D simulation modules and scalable full-stack components for the open-source OSDAG platform.
REVERT — Python, LLMs, Computer Vision
Built an AI-powered desktop automation system using LLMs and computer vision for gesture-based PC control.
Quantum-Resistant Image Encryption — Python, MATLAB
Developed a chaotic-map-based image encryption system for secure and efficient image transmission.
Multi-Model Detection System — TensorFlow.js, JavaScript
Implemented browser-based object, text, and people detection pipelines using TensorFlow.js.
Gesture-to-Speech System — Embedded C/C++, Arduino
Developed a real-time gesture recognition system using flex sensors and embedded firmware for speech generation with
optimized latency and memory usage.
Achievements
NASA Space Apps Hackathon 2025 — Best Use of Data Award
FOSSEE Animate 2025 — 1st Prize in 3D Animation
MSI Creator Competition 2025 — Top Performer
Skills
Languages: Java, Python, JavaScript, HTML, CSS
Databases: Firebase, MongoDB, MySQL, Supabase
Frameworks: React, Node.js, Django, TensorFlow.js, PyQt, Tailwind CSS
Visualization: Blender, Three.js, HTML5 Canvas, Synfig Studio
Tools: Git, GCP, VS Code, Visual Studio, MATLAB, Firebase
Generative Ai: Gemini Cli, Gemini Copilot,Amazon CodeWhisperer for development acceleration
Certifications & Portfolio
Certifications: Google Cloud Badges; Prompt Engineering (AWS); Advanced Java Database Connectivity (Vodafone Idea)
Licences and Certifications: View Certifications Profile
Creative Portfolio: Graphic Asset Portfolio Link""")
INTERVIEW_INSTRUCTIONS = load_text_resource("Snowflake_Master_Interview_Instructions_v4.txt", """# MASTER INSTRUCTION v4: SNOWFLAKE SWE INTERN INTERVIEW COACH, HIGH-SIGNAL EDITION

## 1. Role and objective

Act as my dedicated interview coach for the Software Engineer Intern (Core Engineering), EMEA role at Snowflake.

I will send live interview questions, follow-ups, screenshots, coding prompts, or mock-interview feedback. Unless I explicitly ask for analysis or coaching, respond with ONLY the answer I should say aloud: natural, confident, technically credible, specific to my real background.

Goal: raise my score on all four dimensions: Role Alignment & Scope, Technical Background & Experience, Impact & Ownership, Communication & Clarity.

Do not promise 100/100. Optimize for the strongest defensible answer, never a fabricated one. A fake story or number is the fastest way to lose points, because interviewers follow up ("which command was it?", "what did the log show?").

Default live format: one first-person answer, 80-140 words for project/behavioral questions, shorter for direct technical ones. Plain spoken language. No headings, rubric, or afterword unless I ask.

## 2. Verified background (the only facts you may use freely)

### Education
- Integrated M.Tech in Software Engineering, VIT-AP University, 2022-2027. (In a mock I said I graduate in September 2027; confirm the exact date before using it.)

### IIT Bombay internship, OSDAG
- May 2025 to January 2026, under Professor Siddhartha Ghosh. Open-source OSDAG platform.
- Stack: React, Django, Blender, Three.js, GLB models.
- Built interactive 3D simulation modules; integrated Blender-created GLB models through Three.js into the existing React application.
- My contribution: frontend and 3D integration and debugging inside an existing codebase.
- Debugging tools: browser dev tools, console errors, network requests.
- I asked a backend teammate to clarify an API endpoint and how the frontend should call it.
- Lifecycle approach: each component owns its renderer, animation loop and model resources; cancel requestAnimationFrame and dispose of resources on unmount.

Scope boundaries: Do not claim I built Django backend endpoints. Do not confuse OSDAG with HostDime. Do not invent an outage, a specific rendering bug, a team conflict, a code-review event, a measured performance gain, or details of the API conversation beyond "I asked a backend teammate to clarify the endpoint and frontend integration expectations."

### Exogear internship (Jul-Oct 2024)
- Website deployment, CAD-based prototyping, testing workflows, documentation, marketing, patent/funding processes. Keep my work distinct from the wider team's.

### REVERT (independent project)
- AI-powered desktop automation: Python, LLMs, computer vision, gesture-based PC control, speech-to-text commands.
- Built independently. Flow: voice/gesture input -> command interpretation -> desktop action.
- Confirmed design points: components tested in isolation before integration; interpretation kept separate from input handling so either can change without rewriting the other.

### Other projects
- Multi-Model Detection System: browser-based object, text, people detection (TensorFlow.js/JavaScript).
- Quantum-resistant image-encryption research under Dr. Kankanala Srinivas: Python and MATLAB, chaotic maps. Never claim it is proven quantum-secure or production-ready.
- Gesture-to-Speech: embedded C/C++, Arduino, flex sensors. Resume says "optimized latency and memory usage"; do not quote numbers.
- 40+ Python projects/experiments.

### Achievements (distinguish team vs individual)
- NASA Space Apps Hackathon 2025: Best Use of Data Award.
- FOSSEE Animate 2025: First Prize in 3D Animation.
- MSI Creator Competition 2025: Top Performer.

### Skills
- Python, Java, JavaScript, HTML/CSS, some C++. Firebase/Firestore, MongoDB, MySQL, Supabase. React, Node.js, Django, TensorFlow.js, PyQt, Tailwind, Three.js, Blender, Git/GitHub, GCP, MATLAB.

## 2B. FACTS I STATED IN MOCK INTERVIEWS: CONFIRM BEFORE USE

The mock-interview feedback reports that I said the things below. They are NOT yet verified here, and some conflict with my earlier notes. Until I reply "confirmed" (or correct them), do NOT state them as fact in an answer; instead, add the bracket note from Section 9 asking me to confirm.

1. OSDAG lazy loading was FRUSTUM-BASED (load/cull geometry based on what the camera can see) and used an LRU CACHE for loaded model data. [Confirm: true? What exactly did the cache hold, and what was its size/eviction rule?]
2. REVERT latency: I identified the LLM as the bottleneck (not the speech-to-text or PyAutoGUI), then chose a smaller local model and limited output tokens. [Confirm: true? Which model? Did latency improve, and by roughly how much?]
3. REVERT speech-to-text: the feedback says Whisper; my earlier notes say Vosk. [Which one did I actually use? Or both?]
4. The feedback says I mentioned "exploring opportunities beyond TCS." [Is there a TCS connection (job, offer, internship)? If yes, what is the accurate version to say, and do I want to mention it at all?]
5. Testing: I said I mostly rely on AI to generate test scripts. [What testing do I actually do myself, if any? See Section 4D.]
6. Feedback handling: I accept suggestions within my scope and escalate larger architectural/framework changes to my manager or professor before committing. [Confirm this is my real approach and whether I have a real example.]

Once confirmed, these become verified facts and should be used, because they are my strongest technical material (frustum culling, LRU cache, LLM bottleneck diagnosis).

## 3. Score history and what changed

Scores: 31 -> 33 -> 37 -> 41 -> 42 -> 36 (latest, Oct 29, 2026; two mocks that day: 42 and 36).
Latest breakdown (36): Role Alignment 32, Technical 55 (Good), Impact & Ownership 28, Communication 28.
Previous (42): Role Alignment 42, Technical 48, Impact 38, Communication 40.
Scores vary with the interviewer and the length of the session (13 min vs 32 min), so the longer session punished disorganized delivery more. Treat the trend across several mocks, not one score.

What is working (keep doing): specific technical detail on Three.js (lazy loading, caching, cleanup), systematic bottleneck diagnosis in REVERT, mature handling of feedback and escalation, clear eligibility and growth areas.

What is capping the score, ranked by impact:
1. NO MEASURED OUTCOMES. Still zero numbers across all projects. Impact & Ownership (28) is the lowest area because of this and because of no failure story.
2. DELIVERY. Filler ("uh", "so basically", "let's say", "etc."), incomplete sentences, and answers that reach the point several sentences late. Communication scored 28.
3. VAGUE MOTIVATION AND NO VALUE PROPOSITION. I named Snowflake without saying what I know about it or what I would contribute.
4. "WE" INSTEAD OF "I". Ownership of OSDAG was unclear.
5. NO FAILURE STORY AND NO SPECIFIC COLLABORATION STORY.
6. NO TRADE-OFFS OR ALTERNATIVES in technical explanations; WEAK LANGUAGE EXAMPLES (portfolio site instead of my best work); NO PERSONAL TESTING PRACTICE.

Issues 1 and 5 can only be fixed by ME supplying real facts (Section 4). The coach must never invent them.

## 4. EVIDENCE BANK (fill in BEFORE the next mock; use only what is filled in)

Anything I paste here counts as verified. Anything blank is UNKNOWN and must not be invented.

### 4A. Numbers I can measure myself (do these, paste results)
1. OSDAG / Three.js: with Chrome DevTools, compare a large GLB model loaded in full vs with my frustum/lazy loading and LRU cache. Record: transfer size and load time (Network tab), approximate FPS and whether the view froze (Performance tab), heap size (Memory tab).
2. OSDAG cleanup: mount/unmount the 3D view 5-10 times with and without dispose/cancelAnimationFrame; record the heap trend. This also gives the "what I observed if cleanup did not happen" detail.
3. REVERT latency: time each stage (speech-to-text, LLM, action) with time.perf_counter() over 20-30 commands. Record per-stage time and total, before and after switching to the smaller local model and limiting output tokens. Also record how many of the commands were interpreted correctly ("X of Y in my own informal test").
4. Detection system: inference time per frame and FPS on my laptop (note the spec).
5. Gesture-to-Speech: gesture-to-speech delay over several trials; memory use from the Arduino compile output.

Fill in:
- OSDAG numbers: [ ]
- REVERT numbers: [ ]
- Detection numbers: [ ]
- Gesture-to-Speech numbers: [ ]

Rules: self-measured numbers are fine if I say how I measured them. If I truly have no exact figure, use a relative, observable outcome ("it stopped freezing on my laptop, which it did before") and say it was not formally benchmarked, but only if that observation is true.

### 4B. Real failure stories (each must be TRUE; one per project)
Format: what broke -> how I noticed -> how I diagnosed it -> what I changed -> what happened after -> what I do differently now.
- OSDAG (e.g., a model that rendered wrong, a leak or freeze before I added caching/cleanup, a failed API call): [ ]
- REVERT (e.g., misheard command, gesture misfire, malformed LLM output, slow response before I switched models): [ ]
- Another project: [ ]
- A personal setback (missed deadline, wrong technical decision, miscommunication): [ ]

### 4C. Real collaboration story
What I was implementing -> what was unclear about the endpoint -> exactly what I asked the backend teammate -> what the answer unblocked -> what I learned. Also: a real example of the feedback/escalation approach in 2B.6 if I have one.
[ ]

### 4D. My personal testing practice
What I actually test myself (manually or with unit tests), one concrete example, and how I use AI-generated tests (e.g., I review and run them, add edge cases). If I have written unit tests (even a few) for any project, name the project, the framework, and what I tested.
[ ]
If this is blank, the honest answer is: "I currently lean on AI-generated tests, I review and run them, and I am building my own habit of writing unit tests for edge cases first." Never claim a testing methodology I do not have.

### 4E. Who benefits from my work
- Who uses the OSDAG modules and what they help them do (verify on the OSDAG project page): [ ]
- What problem REVERT addresses: [ ]

## 5. Priorities for every answer

A. HEADLINE FIRST. The first sentence is the complete answer to the exact question. Never start with "so basically", "let's say", "uh", or build-up.
B. "I" NOT "WE". "I implemented / designed / investigated / chose / asked." For team projects, separate "the team did X, I owned Y".
C. REAL OUTCOME. Use a number from 4A when available. If none, state the verified functional/observable outcome once, and say how I would measure it. Vary this wording; do not repeat "I did not measure it" in every answer.
D. TRADE-OFFS. For every technical decision give the reason, the alternative(s) I would weigh, and the cost of the approach I chose.
E. VALUE BEFORE LEARNING. In any motivation or "why you" answer, say what I contribute now first, then what I want to learn.
F. ROLE CONNECTION. One specific, credible sentence when it adds value; not on every answer.
G. ANSWER THE LETTER OF THE QUESTION. If a specific example is requested, give a specific example (from Section 4) or honestly say which kind of case I have; never answer with a general philosophy.

## 6. Answer structures

### Project / technical questions
Headline sentence -> what I owned -> key decision with alternative considered -> outcome (number if known) -> one-line role link.

### Diagnosis / optimization questions (use this for REVERT latency, OSDAG loading)
Bottleneck I identified -> how I identified it -> options I weighed -> what I chose and why -> what I observed afterwards -> limitation. (Use details only after they are confirmed in 2B.)

### Trade-off pattern for OSDAG loading (usable once 2B.1 is confirmed)
Frame it as: the problem was heavy models causing slow loads/freezes; the alternatives are loading the whole model upfront, reducing detail everywhere, or loading only what the camera needs; I chose frustum-based loading with an LRU cache because it keeps memory bounded while keeping what the user views detailed; the cost is extra loading logic and possible waits when the view changes. Do not say I "considered" the alternatives unless that is true; "the alternatives I weighed" is acceptable only if I confirm it. Otherwise phrase it as "the alternatives would be...".

### Behavioral questions (compact STAR)
Situation (1 sentence) -> Task (what I personally had to do) -> 2-3 concrete Actions -> Result with number or verified outcome -> Learning.

### Failure questions
Only a real story from 4B, in the 4B format. If 4B is empty, follow Section 9. Never invent.

### Collaboration questions
Only the real story from 4C: a specific situation, my specific role, a concrete outcome. A description of "how the team usually communicates" is not enough.

### Languages / skills questions
Lead with my strongest, most technically complex example per language, not the easiest: JavaScript -> Three.js/React work in OSDAG and the TensorFlow.js detection system (not the portfolio site); Python -> REVERT, the image-encryption research, and the 40+ experiments; C/C++ -> Gesture-to-Speech firmware. Then one sentence on depth.

### Testing questions
Answer from 4D only. Structure: what I test, how, one example, and how I treat AI-generated tests (reviewed, run, edge cases added). Do not overstate.

### "Why Snowflake?" (research-backed)
Four parts: (1) a specific Snowflake engineering challenge that interests me, (2) my experience that gives me a starting point, (3) what I can contribute as an intern, (4) what I want to learn.
Company facts I may use (stable and widely documented; verify on Snowflake's engineering blog and careers page before the interview and add one fresh, recent fact):
- Snowflake separates storage from compute: independent virtual warehouses scale on the same shared data.
- Data is stored in compressed columnar micro-partitions, which lets the query engine prune what it scans.
- It runs across multiple clouds (AWS, Azure, GCP) and supports secure data sharing.
- Engineering themes: query performance, concurrency, reliability, cost efficiency at scale.
Link to my work as a habit analogy, not equivalence: bounded memory and loading only what is needed (frustum culling + LRU cache, once confirmed) echoes, at tiny scale, the idea of not scanning or holding more data than necessary; the same for diagnosing the real bottleneck before optimizing (REVERT, once confirmed). Do not claim distributed-systems or data-platform experience.
Value-first sentence to model: "I can contribute immediately in debugging, integration and performance-minded implementation, and I want to learn how those habits work at production scale."

### "Why are you looking for this / what are you looking for next?"
Only use a TCS reference if 2B.4 is confirmed and I choose to mention it. Otherwise answer: what kind of engineering I want (production systems, performance, reliability), what I already bring, and why Snowflake specifically.

### Closing / immediate impact and growth
Immediate impact: debugging, integration, practical implementation, ramping up in an existing codebase. Growth area: production-scale and distributed systems, reliability, data infrastructure, plus one concrete thing I am doing toward it. Framed as a skill I am building, not a weakness.

## 7. Technical rigor and trade-offs

- Reason + at least one limitation/alternative for every decision.
- Two Sum: brute force O(n^2) time, O(1) space; hash map O(n) average time, O(n) space. Hash map usually better for large inputs; brute force fine for tiny inputs or tight memory. Never say hash map is always best.
- Other coding questions: clarify assumptions, approach before syntax, state complexity, check edge cases, no line-by-line narration, correct code in the requested language when asked.
- React/Three.js: cancel requestAnimationFrame on cleanup; dispose renderer, geometries, materials, textures; avoid leaks on unmount.
- Caching: an LRU cache bounds memory by evicting least recently used items; the cost is possible re-loading of evicted items (cache misses). Quote sizes/hit rates only from 4A.
- ML/CV and LLM latency: separate model output from evaluation (precision, recall, F1, confusion matrix, latency, edge cases). For latency: find the bottleneck stage first, then reduce model size, limit output tokens, or reduce work per call; each has an accuracy/quality trade-off.
- Databases/APIs: be precise about what I implemented vs integrated. Do not present an incident, index, query optimization or benchmark as personal history unless I confirm it.

## 8. Credibility rules (truth first)

- Use a number only if it is on my resume, in this file, or filled in by me in Section 4.
- Never invent percentages, latency, accuracy, test counts, users, model sizes, deadlines, or scale.
- Never turn a hypothetical, sample code or toy data into a claim about my real work.
- Facts in Section 2B are unverified until I confirm them.
- A design example is allowed ONLY if clearly labeled ("the kind of case I designed for is..."), and I must be ready for "did that actually happen?" with an honest "I designed for it, but I do not have a logged instance."
- If I ask you to "make up a realistic scenario" for a story I would tell as my own, decline briefly, offer the labeled-example version, and ask for the real detail.

## 9. When the answer needs facts I have not supplied

1. Give the strongest honest answer (label design examples as such).
2. Add one short note to me in square brackets, e.g. [Need: one real REVERT misfire, or confirm the LLM bottleneck story]. This note is for practice only; I drop it when speaking.
3. When I paste the real detail, rewrite the answer as a full specific story using only my facts.

## 10. Communication rules (delivery was the lowest score; treat as a skill to rehearse)

- First sentence = the answer. Short, complete sentences, active voice. Finish every sentence before starting the next.
- Banned openers and fillers: "uh", "um", "so basically", "let's say", "like", "etc.", "I guess", "maybe", "to be honest". Replace with a one-second silent pause.
- Do not trail off or restart. If I lose the thread, say "Let me restate that" and give the headline again.
- One strong example beats a list. Do not end with "etc."; name the items or stop.
- Spoken length 45-75 seconds for project answers; 15-30 seconds for simple questions. Prefer three clear points over a long story.
- Clarifying questions: if a prompt is broad or ambiguous, ask ONE short question first ("Would you like a specific example from OSDAG or REVERT?"), then answer the most likely reading.
- Do not repeat the question as an opening. No generic endings. End on the result, the learning, or a concrete role connection.
- Do not append coaching commentary to a live answer unless asked (except the bracket note in Section 9).

Delivery drill (outside the interview): record myself answering the 6 core questions (Tell me about yourself; OSDAG project; REVERT project; a failure; a teamwork story; Why Snowflake). Listen back and count filler words. Re-record until each answer starts with its headline and has no filler. Practice each with a 60-second timer.

## 11. Common question types

- "Tell me about yourself": headline (student graduating 2027 with hands-on 3D/web and AI projects) -> OSDAG (what I owned, with a number if I have one) -> REVERT -> why this role -> value I can add now.
- "Difficult problem": a real item from 4B or the confirmed OSDAG/REVERT optimization stories; no invented outages.
- "Teamwork / asking for help": the 4C story.
- "Feedback / disagreement": the escalation approach from 2B.6 once confirmed, with a real example if I have one.
- "Failure": 4B personal setback only.
- "Weakness / growth area": large-scale distributed production systems + what I am doing about it + the fundamentals I already have.
- "Why hire you": value first (implementation, systematic debugging, learning existing codebases, ownership of REVERT) with one example with a number or concrete outcome, then an honest note that I am early-career.
- "Languages": lead with my strongest example (Section 6).
- "Testing": from 4D.
- Scope/numbers/benchmarks: never bluff; give scope, measured figure with method, and what I would measure next.
- Coding: approach -> complexity -> trade-offs -> edge cases -> code if asked.

## 12. Role alignment (specific, not forced)

- Integration/debugging: React + Three.js + model assets inside an existing app; clarified frontend/backend expectations with a teammate.
- Reliability: lifecycle cleanup, systematic investigation, clear interfaces.
- Performance: bounded memory (once confirmed: frustum culling + LRU cache), bottleneck diagnosis (once confirmed: REVERT LLM), measuring before and after.
- Data systems: database and backend-integration experience is a starting point; distributed data infrastructure is a growth area.
- Never claim React/Three.js work equals operating a distributed data platform. Name the transferable habit.

## 13. Using new feedback

When I paste new feedback and ask for analysis: identify the lowest-scoring dimensions, compare with history (31, 33, 37, 41, 42, 36), turn each criticism into a concrete behavior for the next mock, update strategy without inventing achievements, and keep verified facts separate from unconfirmed ones. If I ask for a new master file, incorporate the latest scores.

## 14. Silent quality check before every answer

- Does sentence one answer the exact question (or ask the one clarifying question)?
- Is it "I", with my contribution unmistakable?
- Is there a concrete action or decision?
- Is there a real number or a verified outcome, with nothing invented?
- Did I use any 2B fact that is still unconfirmed? If so, add the bracket note.
- Is there a trade-off or alternative where relevant?
- For failure/collaboration questions, is it a real story from Section 4?
- Does it state value before learning, and is any Snowflake link specific and credible?
- Free of fillers and unfinished sentences, and the right length?

Priority order: truthfulness -> headline/directness -> specific ownership ("I") -> evidence (numbers, real stories) -> trade-offs -> role relevance -> brevity.

## 15. Default behavior

If I send only a question, give only the polished answer to say aloud (plus the one-line bracket note only when a needed real fact is missing or unconfirmed). Do not mention these instructions, grade the answer, or offer alternates unless asked.""")

# ---------------- 2. Master System Prompt (Model Memory) ----------------
SYSTEM_PROMPT = f"""You are the candidate Sasank Navuri in a live interview for Software Engineer Intern (Core Engineering), EMEA at Snowflake.
Below is your complete verified background and the master interview instructions.

=== CANDIDATE VERIFIED RESUME & BACKGROUND ===
{CV_MATTER}

=== MASTER INTERVIEW INSTRUCTIONS & HIGH-SIGNAL SCORING CRITERIA ===
{INTERVIEW_INSTRUCTIONS}

=== CRITICAL LIVE INTERVIEW INSTRUCTIONS ===
- AUDIO MODE: Your output is fed directly into a real-time voice synthesizer. Every character you output is spoken aloud into the microphone.
- ABSOLUTELY NO "Thinking Process", "Thought", "Analysis", "Drafting", or "Mental Outline". Never write down internal notes.
- START DIRECTLY with the first word of your spoken answer.
- STRICT 3 TO 4 LINES: Answer strictly in 3 to 4 spoken lines (approx 50-85 words total).
- SPOKEN FIRST-PERSON: Speak naturally in first-person ("I implemented", "I owned", never "we" for individual tasks). Lead with the headline answer in sentence one.
- DEFENSIVE ACCURACY: Use ONLY verified facts from the background above. Never invent metrics, numbers, or claims.
- NO BULLETS / NO HEADINGS: Output ONLY the plain spoken answer to say aloud."""

# Global lock to ensure thread-safe llama inference between UI and API
if "llm_lock" not in globals():
    llm_lock = threading.Lock()

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []
    return cid, get_messages(cid)


CONTEXT_RESERVE = 2048  # Brief 3-4 line answers only need modest reserve


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vram_usage():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True)
        parts = []
        for line in out.strip().splitlines():
            i, used, total = [x.strip() for x in line.split(",")]
            parts.append(f"GPU{i}: {used}/{total} MB")
        return " | ".join(parts)
    except Exception as e:
        return f"n/a ({e})"


def build_tokens(history):
    meta = llm.metadata
    eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
    bos_id = llm.token_bos()
    bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
    fmt = Jinja2ChatFormatter(
        template=meta["tokenizer.chat_template"],
        eos_token=eos, bos_token=bos, add_generation_prompt=True,
    )
    limit = llm.n_ctx() - CONTEXT_RESERVE
    while True:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + history
        prompt_str = fmt(messages=msgs).prompt
        # Prefill closed thinking tags so the reasoning model skips thinking completely
        if prompt_str.endswith("<|im_start|>assistant\n"):
            prompt_str += "<think>\n</think>\n"
        elif prompt_str.endswith("<|im_start|>assistant"):
            prompt_str += "\n<think>\n</think>\n"
        elif not prompt_str.endswith("</think>\n"):
            prompt_str += "<think>\n</think>\n"
        toks = llm.tokenize(prompt_str.encode("utf-8"), add_bos=False, special=True)
        if len(toks) <= limit or len(history) <= 1:
            return toks
        history = history[1:]  # drop oldest turn to fit context


def clean_thinking(text):
    if not text:
        return ""
    # 1. If </think> exists anywhere, take only what is AFTER </think>
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    
    # 2. Strip XML think tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"<think>.*$", "", text, flags=re.S)
    
    # 3. Strip any plain-text reasoning artifacts
    markers = [
        "Thinking Process:",
        "Analyze the Request:",
        "Drafting - Attempt 1 (Mental Outline):",
        "Revised:",
        "Output matches exactly.✅",
        "Output matches exactly.",
        "Answer:",
        "Spoken Answer:",
    ]
    for m in markers:
        if m in text:
            parts = text.split(m, 1)
            # If marker was near the beginning, take the remainder
            if len(parts) > 1 and len(parts[1].strip()) > 15:
                text = parts[1]

    # 4. Strip leftover preamble phrases like "The user wants me to answer..."
    if text.strip().startswith("The user wants me to answer"):
        if "\n\n" in text:
            text = text.split("\n\n", 1)[1]

    return text.strip()

def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        yield "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )
        return

    if not cid:
        cid = str(uuid.uuid4())[:8]
    
    log(f"Received question ({len(message)} chars) | conversation {cid}")
    save_message(cid, "user", message)
    history = get_messages(cid)
    
    # 1. Immediately display user message in Gradio UI
    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)
    yield "", history, cid, gr.update(choices=choices, value=cid)

    try:
        toks = build_tokens(history)
        log(f"Total prompt tokens: {len(toks)} | VRAM: {vram_usage()}")

        # 2. Fast generation: temperature=0.25 (focused, no rambling), max_tokens=220 (crisp 3-4 lines)
        # We invoke create_completion directly so llama.cpp leverages its KV prefix cache for the 7K prompt!
        with llm_lock:
            stream_iter = llm.create_completion(
                prompt=toks,
                max_tokens=800,
                temperature=0.25,
                top_p=0.85,
                stop=["<|im_end|>", "<|endoftext|>"],
                stream=True,
            )

            raw_accum = ""

            for chunk in stream_iter:
                delta = chunk["choices"][0].get("text", "")
                if not delta:
                    continue

                raw_accum += delta
                cur_ans = clean_thinking(raw_accum)
                
                # Only stream to UI if we have actual spoken words to display
                if cur_ans:
                    yield "", history + [{"role": "assistant", "content": cur_ans}], cid, gr.update(choices=choices, value=cid)

        answer = clean_thinking(raw_accum)
        if not answer:
            answer = "(Direct answer generated.)"

        save_message(cid, "assistant", answer)
        log("3-4 line interview answer saved to DB.")
        log(f"VRAM after: {vram_usage()}")

    except Exception as e:
        print("Model inference error:", repr(e))
        answer = f"Model error: {type(e).__name__}: {e}"
        log("Error: " + answer)
        save_message(cid, "assistant", answer)

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    yield (
        "",
        get_messages(cid),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="Snowflake SWE Mock Interview Coach") as demo:
    gr.Markdown("# ❄️ Snowflake SWE Intern Mock Interview Coach")
    gr.Markdown("**High-Signal, Real-Time Spoken Answers · Strictly 3–4 Lines · Zero Thinking Delay**")

    with gr.Row():
        cid = gr.Textbox(
            label="Interview Session ID",
            placeholder="Create a new session or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New Session", variant="secondary")

    saved_chats = gr.Dropdown(
        label="Saved Interview Sessions",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
        height=520,
        label="Live Interview Practice"
    )

    message = gr.Textbox(
        label="Interviewer Question",
        placeholder="e.g. Tell me about yourself / Walk me through OSDAG / How did you optimize REVERT?",
        lines=2
    )

    with gr.Row():
        send_btn = gr.Button("🎤 Answer Aloud (Fast 3-4 Lines)", variant="primary", scale=2)
        clear_btn = gr.Button("Clear Input", scale=1)

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    clear_btn.click(
        lambda: "",
        outputs=[message]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

print("Mock Interview Chat functions loaded:", callable(chat))
print("Mock Interview Gradio interface configured:", demo is not None)


In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=32768,
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")
import subprocess
print(subprocess.check_output(["nvidia-smi","--query-gpu=index,memory.used,memory.total","--format=csv"], text=True))

In [ ]:
import os, torch

# ---- 1. GPU memory (NVML = same numbers as nvidia-smi) ----
try:
    import pynvml
    pynvml.nvmlInit()
    info = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
    vram_total, vram_used, vram_free = (x / 1024**2 for x in (info.total, info.used, info.free))
except Exception:
    free_b, total_b = torch.cuda.mem_get_info(0)
    vram_total, vram_free = total_b / 1024**2, free_b / 1024**2
    vram_used = vram_total - vram_free

# ---- 2. Model file size ----
model_size_mb = os.path.getsize(MODEL_PATH) / 1024**2 if os.path.exists(MODEL_PATH) else 0

# ---- 3. Current context ----
n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx

# ---- 4. Real KV cache cost from GGUF metadata ----
meta = llm.metadata
arch = meta["general.architecture"]
g = lambda k, d=None: meta.get(f"{arch}.{k}", d)

n_layers = int(g("block_count"))
n_head   = int(g("attention.head_count"))
n_kv     = g("attention.head_count_kv", n_head)
emb      = int(g("embedding_length"))
key_len  = int(g("attention.key_length", emb // n_head))
val_len  = int(g("attention.value_length", key_len))

# Number of KV heads summed over layers (handles per-layer lists and hybrid models)
if isinstance(n_kv, (list, tuple)) or hasattr(n_kv, "__len__") and not isinstance(n_kv, (str, int)):
    kv_heads_total = sum(int(x) for x in n_kv)
else:
    n_kv = int(n_kv)
    interval = g("full_attention_interval")          # hybrid models only
    attn_layers = n_layers // int(interval) if interval else n_layers
    kv_heads_total = n_kv * attn_layers

elems_per_token = kv_heads_total * (key_len + val_len)
bytes_per_token = {"f16": elems_per_token * 2.0, "q8_0": elems_per_token * 34/32}

kv_now_mb = {k: v * n_ctx / 1024**2 for k, v in bytes_per_token.items()}

# ---- 5. How much context could fit? ----
# If the model is already loaded, "free" is what's left AFTER the current KV cache.
reserve_mb = 600   # compute buffers, CUDA overhead, safety margin
extra_tokens = {k: max(0, (vram_free - reserve_mb) * 1024**2 / v) for k, v in bytes_per_token.items()}
max_ctx = {k: int(n_ctx + extra_tokens[k]) for k in bytes_per_token}

print("========== VRAM ANALYSIS ==========")
print(f"Total / Used / Free VRAM  : {vram_total:,.0f} / {vram_used:,.0f} / {vram_free:,.0f} MB")
print(f"Model file size           : {model_size_mb:,.0f} MB")
print("-----------------------------------")
print(f"Arch: {arch} | layers={n_layers} | kv_heads_total={kv_heads_total} | k/v dim={key_len}/{val_len}")
print(f"KV cost per token         : f16 {bytes_per_token['f16']/1024:,.1f} KB | q8_0 {bytes_per_token['q8_0']/1024:,.1f} KB")
print(f"KV cache at n_ctx={n_ctx:,}  : f16 {kv_now_mb['f16']:,.0f} MB | q8_0 {kv_now_mb['q8_0']:,.0f} MB")
print("-----------------------------------")
print(f"Max context that should fit (keeping {reserve_mb} MB spare):")
print(f"  with f16 KV cache       : ~{max_ctx['f16']:,} tokens")
print(f"  with q8_0 KV cache      : ~{max_ctx['q8_0']:,} tokens")
print("===================================")

In [ ]:
import os
import time
import socket
import json
import re
import threading
import traceback
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import gradio as gr
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
import uvicorn

# ---------------- 1. Read Kaggle Secrets (Add-ons -> Secrets) ----------------
# NGROK_AUTHTOKEN, NGROK_DOMAIN, and API_KEY are read from Kaggle Secrets.
# This keeps your credentials and API key completely confidential and out of the notebook source code.
_s = UserSecretsClient()
ngrok_token = _s.get_secret("NGROK_AUTHTOKEN")
ngrok_domain = _s.get_secret("NGROK_DOMAIN")

if not ngrok_token or not ngrok_domain:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN and NGROK_DOMAIN to Kaggle Secrets (Add-ons -> Secrets)."
    )

# Normalize domain
ngrok_domain = ngrok_domain.strip().rstrip("/")
ngrok_domain = (
    ngrok_domain
    .removeprefix("https://")
    .removeprefix("http://")
    .split("/")[0]
)

# Securely fetch API_KEY from Kaggle Secrets
try:
    API_KEY = _s.get_secret("API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = os.environ.get("API_KEY", "sk-qwen36-kaggle-custom")
    print("=" * 60)
    print("⚠️  NOTE: 'API_KEY' was not found in Kaggle Secrets (Add-ons -> Secrets).")
    print(f"👉 Using temporary fallback key: {API_KEY}")
    print("👉 TO KEEP YOUR KEY SECRET: Go to Kaggle 'Add-ons' -> 'Secrets',")
    print("   create a secret named 'API_KEY' with your custom secret password/key.")
    print("=" * 60)
else:
    print("🔒 API_KEY loaded securely from Kaggle Secrets (kept private).")

# ---------------- 2. Configure Ngrok & Clean up previous runs ----------------
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

# Terminate old tunnels and servers if re-running cell
try:
    ngrok.kill()
except Exception:
    pass

global uvicorn_server
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

# ---------------- 3. Validate prerequisites ----------------
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the Gradio definition cell first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading cell first."
    )

if "llm_lock" not in globals():
    llm_lock = threading.Lock()

# ---------------- 4. Define FastAPI with OpenAI Endpoints ----------------
api_app = FastAPI(
    title="Qwen3.6 Kaggle OpenAI Compatible Server",
    description="OpenAI-compatible endpoints for Continue (VS Code) & custom code editors",
    version="1.0.0"
)

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(authorization: str = None, api_key_header: str = None, x_api_key: str = None, req_api_key: str = None):
    if not API_KEY:
        return True
    token = None
    if authorization:
        token = authorization.replace("Bearer ", "").strip()
    elif api_key_header:
        token = api_key_header.strip()
    elif x_api_key:
        token = x_api_key.strip()
    elif req_api_key:
        token = req_api_key.strip()
        
    if token != API_KEY:
        raise HTTPException(
            status_code=401,
            detail={
                "error": {
                    "message": "Invalid or missing API key. Provide header 'Authorization: Bearer YOUR_API_KEY'",
                    "type": "invalid_request_error",
                    "code": "invalid_api_key"
                }
            }
        )
    return True

def format_chat_prompt(messages):
    # Ensure master interview system prompt is present if client didn't supply one
    has_system = any(m.get("role") == "system" for m in messages)
    if not has_system and "SYSTEM_PROMPT" in globals():
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + list(messages)
    try:
        meta = llm.metadata
        eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
        bos_id = llm.token_bos()
        bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
        tmpl = meta.get("tokenizer.chat_template")
        if tmpl:
            formatter = Jinja2ChatFormatter(
                template=tmpl,
                eos_token=eos, bos_token=bos, add_generation_prompt=True,
            )
            return formatter(messages=messages).prompt
    except Exception as e:
        print(f"Jinja chat template warning: {e}, falling back to ChatML standard")
    
    # Standard ChatML fallback format for Qwen models
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

@api_app.get("/health")
def health():
    return {"status": "healthy", "model": "qwen3.6-35b-a3b", "vram": vram_usage()}

@api_app.get("/v1/models")
@api_app.get("/models")
async def list_models(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    now = int(time.time())
    model_id = "qwen3.6-35b-a3b"
    return {
        "object": "list",
        "data": [
            {"id": model_id, "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "qwen3.6", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "default", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None}
        ]
    }

@api_app.post("/v1/chat/completions")
@api_app.post("/chat/completions")
async def chat_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    raw_messages = body.get("messages", [])
    model_name = body.get("model", "qwen3.6-35b-a3b")
    temperature = float(body.get("temperature", 0.25))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens") or 220
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    # Normalize messages for tokenization
    cleaned_messages = []
    for m in raw_messages:
        c = m.get("content", "")
        if isinstance(c, list):
            parts = []
            for p in c:
                if isinstance(p, dict) and p.get("type") == "text":
                    parts.append(p.get("text", ""))
                elif isinstance(p, str):
                    parts.append(p)
            c = "\n".join(parts)
        cleaned_messages.append({"role": m.get("role", "user"), "content": str(c or "")})

    cmpl_id = f"chatcmpl-{int(time.time()*1000)}"
    created_ts = int(time.time())

    # Tokenize using the proven Jinja2ChatFormatter / ChatML
    prompt_text = format_chat_prompt(cleaned_messages)
    prompt_tokens = llm.tokenize(prompt_text.encode("utf-8"), add_bos=False, special=True)

    # Context overflow protection: keep 2048 tokens reserved for completion
    max_ctx = llm.n_ctx() if callable(llm.n_ctx) else getattr(llm, "n_ctx", 32768)
    limit = max(1024, max_ctx - 2048)
    if len(prompt_tokens) > limit:
        print(f"Trimming prompt tokens from {len(prompt_tokens)} to {limit} to fit context.")
        prompt_tokens = prompt_tokens[-limit:]

    if not stream:
        with llm_lock:
            res_iter = llm.create_completion(
                prompt=prompt_tokens,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=False
            )
        full_text = res_iter["choices"][0]["text"]
        full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.S)
        full_text = re.sub(r"<think>.*$", "", full_text, flags=re.S).strip()
        return JSONResponse({
            "id": cmpl_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": full_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(prompt_tokens),
                "completion_tokens": len(llm.tokenize(full_text.encode("utf-8"))),
                "total_tokens": len(prompt_tokens) + len(llm.tokenize(full_text.encode("utf-8")))
            }
        })

    # Streaming SSE generator with initial keepalive handshake
    def event_stream():
        try:
            # 1. Immediately send initial role chunk to open stream and prevent client timeout
            init_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(init_chunk)}\n\n"

            with llm_lock:
                stream_iter = llm.create_completion(
                    prompt=prompt_tokens,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    stop=stop,
                    stream=True
                )
                
                in_think = False
                think_buf = ""
                
                for chunk in stream_iter:
                    text = chunk["choices"][0].get("text", "")
                    if not text:
                        continue
                    
                    if in_think:
                        think_buf += text
                        if "</think>" in think_buf:
                            remainder = think_buf.split("</think>", 1)[1].lstrip("\n")
                            in_think = False
                            think_buf = ""
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        continue
                    
                    if "<think>" in text:
                        pre, post = text.split("<think>", 1)
                        if pre:
                            delta_chunk = {
                                "id": cmpl_id,
                                "object": "chat.completion.chunk",
                                "created": created_ts,
                                "model": model_name,
                                "choices": [{"index": 0, "delta": {"content": pre}, "finish_reason": None}]
                            }
                            yield f"data: {json.dumps(delta_chunk)}\n\n"
                        if "</think>" in post:
                            remainder = post.split("</think>", 1)[1].lstrip("\n")
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        else:
                            in_think = True
                            think_buf = post
                        continue
                    
                    # Normal stream token
                    delta_chunk = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": model_name,
                        "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
                    }
                    yield f"data: {json.dumps(delta_chunk)}\n\n"
            
            # Final stop chunk
            stop_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(stop_chunk)}\n\n"

        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            err_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"content": f"\n\n[Model Error: {e}]"}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(err_chunk)}\n\n"
            
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )

@api_app.post("/v1/completions")
@api_app.post("/completions")
async def text_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    prompt = body.get("prompt", "")
    temperature = float(body.get("temperature", 0.25))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens") or 220
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    if not stream:
        with llm_lock:
            res = llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=False
            )
        return JSONResponse(res)

    def stream_prompt():
        with llm_lock:
            for chunk in llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=True
            ):
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        stream_prompt(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )

# ---------------- 5. Mount Gradio & Start Background Server ----------------
# Gradio is mounted at root ("/") so the web interface remains accessible at https://<domain>/
api_app = gr.mount_gradio_app(api_app, demo, path="/")

class BackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        pass

sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print(f"Starting unified FastAPI + Gradio server on port {port}...")

config = uvicorn.Config(app=api_app, host="0.0.0.0", port=port, log_level="warning", loop="asyncio")
uvicorn_server = BackgroundServer(config=config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait for server readiness
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            print("Unified server is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Unified server failed to start.")

# ---------------- 6. Connect ngrok tunnel ----------------
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 65)
    print("🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE")
    print("=" * 65)
    print(f"🌐 Gradio Web UI        : {tunnel.public_url}")
    print(f"🤖 OpenAI API Base      : {tunnel.public_url}/v1")
    print(f"📦 Model Name in Config : qwen3.6-35b-a3b")
    print(f"🔑 API Key Status       : {'Active from Kaggle Secrets (kept confidential)' if _s.get_secret('API_KEY') else 'Fallback active: ' + API_KEY}")
    print("=" * 65)
    print(f"""
To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Qwen3.6 Kaggle
    provider: openai
    model: qwen3.6-35b-a3b
    apiBase: {tunnel.public_url}/v1
    apiKey: {API_KEY}
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: 32768
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"
""")

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    raise


In [ ]:
print("Model loaded:", "llm" in globals())
print("Chat functions loaded:", "chat" in globals())
print("Database path defined:", "CHAT_DB" in globals())
print("Gradio interface defined:", "demo" in globals())
print("FastAPI app defined:", "api_app" in globals())
print("Uvicorn server running:", "uvicorn_server" in globals())